# Passo 4: variáveis explicativas e separação treino/teste

Regra de ouro: **cada variável do mês t usa só dados até o mês t** (sem olhar o futuro).

A separação é **por data**, simulando o uso real do modelo em jan/2025:
- **Treino:** meses-base de jul/2022 a jun/2024 (alvos já conhecidos em jan/2025).
- **Intervalo (fora):** jul/2024 a dez/2024, cujos alvos ainda não seriam conhecidos.
- **Teste:** meses-base de jan/2025 a jun/2025.

In [1]:
import pandas as pd

from src.dados import PASTA_PROCESSED, CHAVES_RECORTE

# Painel do passo 3, com a taxa de inadimplência e o alvo
painel = pd.read_parquet(PASTA_PROCESSED / "painel_alvo.parquet")

# Ordena por recorte e data: os shift() abaixo dependem dessa ordem
painel = painel.sort_values(CHAVES_RECORTE + ["data_base"]).reset_index(drop=True)
print(painel.shape)

(18528, 14)


In [2]:
carteira = painel["carteira_ativa"]

# Taxas do mês, em % da carteira ativa
painel["taxa_atraso_15_90"] = 100 * painel["vencido_de_15_ate_90_dias"] / carteira
painel["taxa_problematico"] = 100 * painel["ativo_problematico"] / carteira

# Agrupa por recorte para que shift(h) pegue o valor de h meses ANTES
# do mesmo recorte (h positivo = passado, nunca o futuro)
grupos = painel.groupby(CHAVES_RECORTE, observed=True)

for h in (3, 6):
    # Variação da inadimplência em pontos percentuais
    painel[f"var_taxa_inad_{h}m"] = painel["taxa_inad"] - grupos["taxa_inad"].shift(h)
    # Crescimento da carteira em %
    painel[f"cresc_carteira_{h}m"] = 100 * (carteira / grupos["carteira_ativa"].shift(h) - 1)

# Variação do atraso curto em 3 meses
painel["var_atraso_15_90_3m"] = (
    painel["taxa_atraso_15_90"] - grupos["taxa_atraso_15_90"].shift(3)
)

VARIAVEIS_NUM = [
    "taxa_inad",
    "taxa_atraso_15_90",
    "taxa_problematico",
    "var_taxa_inad_3m",
    "var_taxa_inad_6m",
    "var_atraso_15_90_3m",
    "cresc_carteira_3m",
    "cresc_carteira_6m",
]
VARIAVEIS_CAT = ["uf", "cliente", "modalidade"]

painel[VARIAVEIS_NUM].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
taxa_inad,18528.0,2.59,4.76,0.00,0.15,1.01,3.71,64.71
taxa_atraso_15_90,18528.0,0.72,1.76,0.00,0.06,0.21,0.99,44.79
taxa_problematico,18528.0,5.44,7.59,0.00,0.96,3.27,7.81,85.78
var_taxa_inad_3m,17370.0,0.09,1.91,-51.35,-0.05,0.00,0.23,37.50
var_taxa_inad_6m,16212.0,0.18,2.58,-54.26,-0.07,0.02,0.42,39.04
var_atraso_15_90_3m,17370.0,0.02,1.15,-39.67,-0.03,0.00,0.06,32.62
cresc_carteira_3m,17370.0,4.41,32.72,-100.00,0.15,3.03,5.96,2068.33
cresc_carteira_6m,16212.0,8.65,45.10,-100.00,1.57,6.19,11.05,1981.59


In [3]:
# Mantém só as linhas com todas as variáveis e com alvo:
# saem os 6 primeiros meses (sem histórico) e os 6 últimos (sem alvo)
base = painel.dropna(subset=VARIAVEIS_NUM + ["alvo"]).copy()
base["alvo"] = base["alvo"].astype(int)

# Separação por data
FIM_TREINO = pd.Timestamp("2024-06-30")
INICIO_TESTE = pd.Timestamp("2025-01-01")

base["parte"] = "intervalo"
base.loc[base["data_base"] <= FIM_TREINO, "parte"] = "treino"
base.loc[base["data_base"] >= INICIO_TESTE, "parte"] = "teste"

resumo = base.groupby("parte").agg(
    linhas=("alvo", "size"),
    primeiro_mes=("data_base", "min"),
    ultimo_mes=("data_base", "max"),
    pct_alvo_1=("alvo", "mean"),
)
resumo["pct_alvo_1"] = (100 * resumo["pct_alvo_1"]).round(1)
resumo.loc[["treino", "intervalo", "teste"]]

,linhas,primeiro_mes,ultimo_mes,pct_alvo_1
parte,,,,
treino,9264,2022-07-31,2024-06-30,13.0
intervalo,2316,2024-07-31,2024-12-31,23.6
teste,2316,2025-01-31,2025-06-30,24.9


In [4]:
# Correlação de cada variável numérica com o alvo, só no treino
# (olhar o teste aqui seria uma forma de "colar")
treino = base[base["parte"] == "treino"]
print(treino[VARIAVEIS_NUM].corrwith(treino["alvo"]).round(3).sort_values(ascending=False))

# Salva a base do modelo; o passo 5 filtra pela coluna "parte"
colunas = ["data_base"] + VARIAVEIS_CAT + VARIAVEIS_NUM + ["alvo", "parte"]
base[colunas].to_parquet(PASTA_PROCESSED / "base_modelo.parquet", index=False)
print("Base salva:", base[colunas].shape)

taxa_atraso_15_90      0.137
var_atraso_15_90_3m    0.108
taxa_problematico      0.084
var_taxa_inad_3m       0.026
taxa_inad              0.011
cresc_carteira_6m     -0.005
cresc_carteira_3m     -0.007
var_taxa_inad_6m      -0.008
dtype: float64
Base salva: (13896, 14)
